# Stage 3 prep: dense checkpoints for all 10 seeds

1. **Runtime -> Change runtime type -> T4 GPU -> Save.**
2. **Runtime -> Run all.**
3. When an upload button appears under cell 2, choose `grokking-dynamics.zip`.
4. Keep this tab open. Training takes about **45 minutes**.
5. The last cell downloads **10 zips, one per seed** (~107 MB each, ~1.07 GB total).
   One per seed so a dropped download only costs that seed. If a download fails, re-run
   just that line, or grab the file from the Files pane on the left (`dist/`).

This re-trains baseline seeds 0-9 with the denser checkpoint schedule
(`checkpoint_every: 250` up to step 15000). It uses `regenerate_checkpoints.py`, which
writes its metrics to a temp file and **byte-compares them with the committed CSVs**, so
the run also proves that dense checkpointing did not perturb training. Expect 10x MATCH.
Committed CSVs in `results/` are never overwritten. The wd=0 control is not re-run: it
never groks, so it has no transition to resolve.

In [ ]:
# 1. Check that a GPU is attached
import torch
assert torch.cuda.is_available(), 'No GPU: Runtime > Change runtime type > T4 GPU, then Run all again.'
print(torch.__version__, 'CUDA', torch.version.cuda, '-', torch.cuda.get_device_name(0))

In [ ]:
# 2. Upload and unpack the code (choose grokking-dynamics.zip)
import os, zipfile
os.chdir('/content')
if not os.path.isdir('/content/grokking-dynamics'):
    from google.colab import files
    uploaded = files.upload()
    zipfile.ZipFile(next(iter(uploaded))).extractall('/content')
os.chdir('/content/grokking-dynamics')
print(sorted(os.listdir()))

In [ ]:
# 3. Install dependencies and record the environment
!pip install -q 'transformer_lens>=2.0,<4' pytest pyyaml
!mkdir -p logs dist
!(nvidia-smi --query-gpu=name,driver_version --format=csv,noheader; python --version; python -c "import torch, numpy, importlib.metadata as m; print('torch', torch.__version__, 'cuda', torch.version.cuda); print('numpy', numpy.__version__); print('transformer_lens', m.version('transformer_lens'))") | tee logs/ENVIRONMENT_stage3.txt

In [ ]:
# 4. Show the schedule this config will produce, before spending 45 minutes on it
import sys, yaml
sys.path.insert(0, '.')
from src.train import checkpoint_steps, log_checkpoint_steps
t = yaml.safe_load(open('configs/baseline.yaml'))['train']
full = log_checkpoint_steps(t['num_steps'], t['num_checkpoints'])
every = checkpoint_steps(t['num_steps'], t['num_checkpoints'], t.get('checkpoint_every'), t.get('checkpoint_every_until'))
n_dense = len(every) - len(full)
mb = len(full) * 2.6 + n_dense * 0.87
print(f'{len(every)} checkpoints per seed = {len(full)} full + {n_dense} weights-only')
print(f'approx {mb:.0f} MB per seed, {mb * 10 / 1024:.2f} GB for 10 seeds')
print('first 12:', every[:12])
print('around the grok:', [s for s in every if 4000 <= s <= 7000])

# Fail fast. regenerate_checkpoints.py verifies by byte-comparing against the committed
# CSVs; without them it prints NO COMMITTED CSV and exits 1 only AFTER training. That is
# how a 45-minute run once verified nothing (2026-09-15).
import glob
n_csv = len(glob.glob('results/01_baseline/baseline_seed*.csv'))
assert n_csv >= 10, (
    f'Only {n_csv} committed CSVs in this bundle. Rebuild it locally with '
    'python colab/make_bundle.py, re-upload, and run again - otherwise cell 6 verifies nothing.'
)
print(f'{n_csv} committed CSVs present - the MATCH check in cell 6 will be meaningful')

In [ ]:
# 5. Tests before training (schedule, weights-only checkpoints, and determinism on GPU)
!python -m pytest tests -q -p no:warnings 2>&1 | tee logs/pytest_before_stage3.log

In [ ]:
# 6. Re-train seeds 0-9 with dense checkpoints (~45 min). Expect 10 lines of MATCH.
# MISMATCH would mean checkpointing perturbed training - stop and report it, do not proceed.
!python experiments/regenerate_checkpoints.py --config configs/baseline.yaml --seeds 0-9 --jobs 2 2>&1 | tee logs/regen_dense.log

In [ ]:
# 7. Inventory: how many checkpoints landed per seed, and how big
import os, glob
total = 0
for s in range(10):
    d = f'checkpoints/01_baseline/baseline_seed{s}'
    files_ = glob.glob(d + '/step*.pt')
    size = sum(os.path.getsize(f) for f in files_) / 1048576
    total += size
    print(f'seed {s}: {len(files_):4d} checkpoints, {size:7.1f} MB')
print(f'total {total / 1024:.2f} GB')

In [ ]:
# 7b. Verify the checkpoints ARE the committed run, by recomputing metrics from them.
# Cell 6 proves the metrics CSV reproduces; this proves the saved weights match it, which
# is the artifact Stage 3 actually consumes. Writes a small CSV to compare against locally.
import glob, csv, os
import torch
from src.train import load_checkpoint, evaluate, param_norm, load_config, read_metrics
from src.data import make_data

torch.backends.cuda.matmul.allow_tf32 = False  # must match how the CSVs were produced
torch.backends.cudnn.allow_tf32 = False

cfg = load_config('configs/baseline.yaml')
d = cfg['data']
data = make_data(d['p'], d['train_frac'], d['split_seed'])
tx, ty, ex, ey = (t.to('cuda') for t in (data.train_x, data.train_y, data.test_x, data.test_y))

rows, worst = [], 0.0
for seed in range(10):
    committed = read_metrics(f'results/01_baseline/baseline_seed{seed}.csv')
    by_step = {int(s): i for i, s in enumerate(committed['step'])}
    for path in sorted(glob.glob(f'checkpoints/01_baseline/baseline_seed{seed}/step*.pt')):
        step = int(os.path.basename(path)[4:10])
        if step % 100:  # only steps that also have a committed CSV row
            continue
        model, _ = load_checkpoint(path)
        model = model.to('cuda').eval()
        trl, tra = evaluate(model, tx, ty)
        tel, tea = evaluate(model, ex, ey)
        pn = param_norm(model)
        rows.append([seed, step, trl, tel, tra, tea, pn])
        i = by_step[step]
        worst = max(worst, abs(tea - committed['test_acc'][i]), abs(pn - committed['param_norm'][i]) / pn)
    print('seed', seed, 'checked', flush=True)

with open('logs/ckpt_metrics.csv', 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['seed', 'step', 'train_loss', 'test_loss', 'train_acc', 'test_acc', 'param_norm'])
    w.writerows([r[:2] + [repr(float(v)) for v in r[2:]] for r in rows])

print(f'{len(rows)} checkpoints verified against the committed CSVs')
print(f'worst disagreement (test acc / relative param norm): {worst:.3e}')
assert worst < 1e-6, 'checkpoints do not match the committed run - stop and report this'

In [ ]:
# 8. Package: one zip per seed, plus a small zip of logs
import subprocess
for s in range(10):
    out = f'dist/ckpt_seed{s}.zip'
    subprocess.run(['rm', '-f', out], check=False)
    subprocess.run(['zip', '-q', '-r', out, f'checkpoints/01_baseline/baseline_seed{s}'], check=True)
subprocess.run(['rm', '-f', 'dist/stage3_logs.zip'], check=False)
subprocess.run(['zip', '-q', '-r', 'dist/stage3_logs.zip', 'logs'], check=True)
!ls -lh dist

In [ ]:
# 9. Download. If your browser blocks repeated downloads, allow them and re-run this cell;
# already-downloaded files can be skipped by editing the range.
from google.colab import files
files.download('dist/stage3_logs.zip')
for s in range(10):
    files.download(f'dist/ckpt_seed{s}.zip')